# Models vs observations: susceptibility and the closure diameter

**Question.** Do NorESM2 (OsloAero) and EC-Earth3 (TM5-M7) put their grid cells in the same part of the
S(100 nm)–D<sub>SB</sub> and S(D<sub>SB</sub>)–D<sub>SB</sub> planes as the observations, globally and at the
observed locations?

Definitions are the same for models and observations:

* **S(D)** = binned-median log-log slope of N<sub>d</sub> (CDNC) on N(>D) (CCN above the cutoff), the canonical
  recipe (log10 → equal-number bins → bin medians → least squares).
* **D<sub>SB</sub>** (= D<sub>Δ</sub> in the observational scripts) = cutoff where mean log N<sub>d</sub> = mean log N(>D),
  i.e. where the squared bias SB = Δ² is zero; found by interpolating the zero crossing of Δ(D) in diameter.
* **S(D<sub>SB</sub>)** = S(D) interpolated (in log D) to D<sub>SB</sub>.

Inputs (all on Arrhenius):

| | file | content |
|---|---|---|
| NorESM2 | `py/global_susc_fields.nc` | `slope`, `delta` (radius, lat, lon), `n`; pooled over time and the 10 lowest levels, FREQL > 0.8 |
| EC-Earth3 | `ecearth_global/susceptibility/ecearth_susc_pooled.nc` | `slope` (lat, lon, radius), pooled over time and level |
| EC-Earth3 | `ecearth_global/susceptibility/ecearth_dsb_fields.nc` | `delta` (radius, lat, lon), `D_SB` |
| obs | `results/s_at_ddelta.csv` (from `plot_s_at_ddelta.py`) | S(D<sub>Δ</sub>), D<sub>Δ</sub>, S(100) per source; S(100) CIs from each source's quicklook table |

**Caveats.** Model values pool time and level (cloud layer), the observations are hourly or per pass near
cloud base at one place; model CCN is from the size distribution integrated above the cutoff with the
non-hygroscopic modes excluded, like a dry size distribution. Aircraft campaigns are compared with the mean over
a box around the flight area.

## 1 · Settings

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import LogNorm
from matplotlib.ticker import FixedLocator, ScalarFormatter, NullFormatter

BASE = Path(os.environ.get("PETCHU", "/nobackup/proj/disk/bolinc/personal/petchu"))
NOR_FIELDS = Path(os.environ.get("NOR_FIELDS", BASE / "py/global_susc_fields.nc"))
EC_SUSC    = Path(os.environ.get("EC_SUSC", BASE / "ecearth_global/susceptibility/ecearth_susc_pooled.nc"))
EC_DSB     = Path(os.environ.get("EC_DSB", BASE / "ecearth_global/susceptibility/ecearth_dsb_fields.nc"))
OBS_CSV    = Path(os.environ.get("OBS_CSV", "results/s_at_ddelta.csv"))
FIG_DIR    = Path(os.environ.get("FIG_DIR", BASE / "figures/model_obs"))
FIG_DIR.mkdir(parents=True, exist_ok=True)

RADII_ARE_RADII = True      # model files store cutoff RADIUS in nm; diameter = 2 x radius
D_REF = 100.0               # nm, reference cutoff for S(100)
MIN_N = 500                 # model cells need at least this many cloudy samples (as in the global passes)
SHOW_QUALITY = ("robust", "moderate", "slope only", "low")   # observation qualities to plot
MODELS = {"NorESM2": "#2b6cb0", "EC-Earth3": "#c05621"}       # one colour per model
REG_COL = {"Marine": "#1f77b4", "Marine CAO": "#17becf", "Boreal": "#2ca02c", "Arctic": "#9467bd", "Continental": "#d62728"}
MK = {"Ground NDROP": "o", "Ground in situ": "s", "Cloudnet": "^", "Aircraft": "D", "Ship in situ": "P"}
QA = {"robust": 1.0, "moderate": 0.85, "slope only": 0.5, "low": 0.3}

## 2 · Model fields: S(D), Δ(D) → S(100), D<sub>SB</sub>, S(D<sub>SB</sub>) per grid cell

`dsb_from_delta` is the same function as `py/dsb_tools.py`, so D<sub>SB</sub> is found exactly as in the
earlier NorESM/EC-Earth comparison.

In [ ]:
def dsb_from_delta(delta, diam):
    """D_SB per cell from delta (n_cutoff, ...): first change of sign from negative to positive, linear in D.
    Returns (D_SB, flag) with flag 0 = found, -1 = delta > 0 everywhere, +1 = delta < 0 everywhere, 9 = no data."""
    d = np.asarray(delta, float)
    shp = d.shape[1:]
    d = d.reshape(d.shape[0], -1)
    dsb = np.full(d.shape[1], np.nan)
    flag = np.full(d.shape[1], 9, np.int8)
    good = np.all(np.isfinite(d), axis=0)
    neg, pos = d < 0, d >= 0
    flag[good & np.all(pos, axis=0)] = -1
    flag[good & np.all(neg, axis=0)] = 1
    cross = neg[:-1] & pos[1:]
    has = good & cross.any(axis=0)
    j = np.argmax(cross, axis=0)
    c = np.nonzero(has)[0]
    jj = j[c]
    d0, d1 = d[jj, c], d[jj + 1, c]
    dsb[c] = diam[jj] + (0 - d0) * (diam[jj + 1] - diam[jj]) / (d1 - d0)
    flag[c] = 0
    return dsb.reshape(shp), flag.reshape(shp)


def interp_logd(field, diam, target):
    """field (n_cutoff, ny, nx) evaluated at target (ny, nx) cutoff diameters, linear in log D."""
    f = field.reshape(field.shape[0], -1)
    t = np.asarray(target, float).reshape(-1)
    out = np.full(t.size, np.nan)
    ld = np.log(diam)
    for k in np.nonzero(np.isfinite(t))[0]:
        y = f[:, k]
        ok = np.isfinite(y)
        if ok.sum() >= 2 and diam[ok][0] <= t[k] <= diam[ok][-1]:
            out[k] = np.interp(np.log(t[k]), ld[ok], y[ok])
    return out.reshape(np.shape(target))


def as_rll(da):
    """Transpose to (cutoff, lat, lon); return values, diameters (nm), lat, lon (-180..180, sorted)."""
    rdim = [d for d in da.dims if d not in ("lat", "lon")][0]
    da = da.transpose(rdim, "lat", "lon")
    lon = ((da.lon.values + 180.0) % 360.0) - 180.0
    da = da.assign_coords(lon=lon).sortby("lon").sortby("lat")
    r = da[rdim].values.astype(float)
    return da.values.astype(float), (2.0 * r if RADII_ARE_RADII else r), da.lat.values, da.lon.values


def model_fields(slope_da, delta_da, n_da=None):
    s, diam, lat, lon = as_rll(slope_da)
    d, diam_d, lat_d, lon_d = as_rll(delta_da)
    assert np.allclose(diam, diam_d) and np.allclose(lat, lat_d) and np.allclose(lon, lon_d), "grids differ"
    if n_da is not None:
        n = n_da.assign_coords(lon=((n_da.lon.values + 180) % 360) - 180).sortby("lon").sortby("lat").values
        s[:, n < MIN_N] = np.nan
        d[:, n < MIN_N] = np.nan
    dsb, flag = dsb_from_delta(d, diam)
    return dict(S100=interp_logd(s, diam, np.full(dsb.shape, D_REF)), DSB=dsb, flag=flag,
                S_DSB=interp_logd(s, diam, dsb), lat=lat, lon=lon, diam=diam,
                w=np.broadcast_to(np.cos(np.deg2rad(lat))[:, None], dsb.shape))


nor = xr.open_dataset(NOR_FIELDS)
ecs = xr.open_dataset(EC_SUSC)
ecd = xr.open_dataset(EC_DSB)
M = {"NorESM2": model_fields(nor["slope"], nor["delta"], nor.get("n")),
     "EC-Earth3": model_fields(ecs["slope"], ecd["delta"], ecd.get("n"))}

def wq(v, w, qs=(0.25, 0.5, 0.75)):
    m = np.isfinite(v)
    o = np.argsort(v[m]); cw = np.cumsum(w[m][o]); cw /= cw[-1]
    return [v[m][o][np.searchsorted(cw, q)] for q in qs]

rows = []
for k, m in M.items():
    f = m["flag"]
    rows.append({"model": k, "cells": f.size, "D_SB found %": 100 * np.mean(f == 0),
                 "below range %": 100 * np.mean(f == -1), "above range %": 100 * np.mean(f == 1),
                 **{f"S100 p{int(q*100)}": v for q, v in zip((0.25, 0.5, 0.75), wq(m["S100"], m["w"]))},
                 **{f"D_SB p{int(q*100)}": v for q, v in zip((0.25, 0.5, 0.75), wq(m["DSB"], m["w"]))},
                 **{f"S(D_SB) p{int(q*100)}": v for q, v in zip((0.25, 0.5, 0.75), wq(m["S_DSB"], m["w"]))}})
pd.DataFrame(rows).set_index("model").round(2)

## 3 · Observations

`results/s_at_ddelta.csv` comes from `plot_s_at_ddelta.py` (rerun it after adding sources). The S(100) interval
is read from each source's quicklook table. Approximate locations: ground sites and ships at the site (nearest
model cell); aircraft campaigns as the box around their flight area (mean over cells, area weighted).

In [ ]:
# name -> (lat, lon, half-width in degrees; 0 = nearest cell). Aircraft boxes cover the main flight areas.
LOC = {
    "ACTIVATE winter": (36.5, -72.0, 4), "ACTIVATE summer": (36.5, -72.0, 4), "ACE-ENA": (39.1, -28.0, 3),
    "NAAMES": (48.0, -40.0, 8), "CAMP2Ex": (15.0, 122.0, 5), "ORACLES": (-12.0, 5.0, 7), "ACAPEX": (36.0, -126.0, 5),
    "ACME-V": (70.5, -150.0, 3), "HI-SCALE": (36.6, -97.5, 1.5), "CACTI": (-32.1, -64.7, 1.5), "GoAmazon": (-3.2, -60.6, 1.5),
    "ENA (UHSAS)": (39.09, -28.03, 0), "ENA (SMPS 2022-26)": (39.09, -28.03, 0), "ENA (Cloudnet, SMPS)": (39.09, -28.03, 0),
    "SGP": (36.61, -97.49, 0), "SGP (Cloudnet)": (36.61, -97.49, 0), "LASIC (SMPS)": (-7.97, -14.35, 0),
    "EPCAPE pier": (32.87, -117.26, 0), "EPCAPE Mt Soledad": (32.84, -117.25, 0),
    "Puijo": (62.91, 27.66, 0), "Pallas": (67.97, 24.12, 0), "Zeppelin": (78.91, 11.89, 0), "Hyytiala": (61.85, 24.29, 0),
    "Kenttarova": (67.99, 24.24, 0), "Norunda": (60.09, 17.48, 0), "Mace Head": (53.33, -9.90, 0),
    "Ny-Alesund": (78.92, 11.92, 0), "Cabauw": (51.97, 4.93, 0), "Granada": (37.16, -3.61, 0), "Bucharest": (44.35, 26.03, 0),
    "Melpitz": (51.53, 12.93, 0), "Schneefernerhaus": (47.42, 10.98, 0), "Palaiseau": (48.71, 2.21, 0),
    "Oliktok": (70.50, -149.89, 0), "Bankhead": (34.34, -87.34, 0), "Maido": (-21.08, 55.38, 0), "Andoya (Cloudnet)": (69.28, 16.01, 0),
    "AO2018 fog (FSSP)": (88.5, 30.0, 2), "AO2018 fog (residuals)": (88.5, 30.0, 2), "AO2018 (Cloudnet)": (88.5, 30.0, 2),
    "MOSAiC (Cloudnet, SMPS)": (86.0, 60.0, 4),
}

obs = pd.read_csv(OBS_CSV)
obs = obs[obs.quality.isin(SHOW_QUALITY)].copy()

def s100_ci(path):
    try:
        t = pd.read_csv(path)
        i = int(np.argmin(np.abs(t.D_nm.to_numpy(float) - D_REF)))
        return float(t.S_lo.iloc[i]), float(t.S_hi.iloc[i])
    except Exception:
        return np.nan, np.nan

obs[["S100_lo", "S100_hi"]] = [s100_ci(f) for f in obs.file]
missing = sorted(set(obs.label) - set(LOC))
if missing:
    print("no location for:", missing, "- add them to LOC to get model values at the site")
obs[["label", "regime", "method", "quality", "n", "D_Delta", "S_at_D", "S100", "S100_lo", "S100_hi"]].round(2)

## 4 · Model values at the observed locations

In [ ]:
def at_site(m, lat, lon, hw):
    """Area-weighted mean of S100, D_SB (geometric) and S(D_SB) over the box, or the nearest cell if hw == 0."""
    if hw == 0:
        iy = int(np.abs(m["lat"] - lat).argmin())
        ix = int(np.abs(((m["lon"] - lon) + 180) % 360 - 180).argmin())
        sel = (slice(iy, iy + 1), slice(ix, ix + 1))
        box = np.zeros(m["DSB"].shape, bool); box[sel] = True
    else:
        dl = np.abs(((m["lon"][None, :] - lon) + 180) % 360 - 180)
        box = (np.abs(m["lat"][:, None] - lat) <= hw) & (dl <= hw * max(1.0, 1 / max(np.cos(np.deg2rad(lat)), 0.2)))
    out = {}
    for k in ("S100", "DSB", "S_DSB"):
        v, w = m[k][box], m["w"][box]
        ok = np.isfinite(v)
        if not ok.any():
            out[k] = np.nan
        elif k == "DSB":
            out[k] = float(np.exp(np.average(np.log(v[ok]), weights=w[ok])))
        else:
            out[k] = float(np.average(v[ok], weights=w[ok]))
    out["cells"] = int(np.isfinite(m["DSB"][box]).sum())
    return out

site = []
for _, o in obs.iterrows():
    if o.label not in LOC:
        continue
    lat, lon, hw = LOC[o.label]
    for k, m in M.items():
        v = at_site(m, lat, lon, hw)
        site.append({"label": o.label, "regime": o.regime, "method": o.method, "quality": o.quality, "model": k,
                     "obs_S100": o.S100, "obs_D": o.D_Delta, "obs_S_D": o.S_at_D,
                     "mod_S100": v["S100"], "mod_D": v["DSB"], "mod_S_D": v["S_DSB"], "cells": v["cells"]})
site = pd.DataFrame(site)
site.to_csv(FIG_DIR / "model_at_obs_sites.csv", index=False, float_format="%.3f")
site.pivot_table(index="label", columns="model", values=["mod_S100", "mod_D", "mod_S_D"]).round(2)

## 5 · Figure: models (density of grid cells) behind the observations

Rows: S(100) vs D<sub>SB</sub>, and S(D<sub>SB</sub>) vs D<sub>SB</sub>. Columns: NorESM2, EC-Earth3. The grey shading
is the area-weighted density of model grid cells (sequential, log scale); observations are drawn as in the
synthesis figures (colour = regime, marker = method, fainter = lower quality); small open circles in the model
colour are the model at each observed location.

In [ ]:
def density(ax, x, y, w, xlim, ylim, nb=60):
    ok = np.isfinite(x) & np.isfinite(y) & (x > 0)
    xe = np.logspace(np.log10(xlim[0]), np.log10(xlim[1]), nb + 1)
    ye = np.linspace(ylim[0], ylim[1], nb + 1)
    H, _, _ = np.histogram2d(x[ok], y[ok], bins=[xe, ye], weights=w[ok])
    H = np.where(H > 0, H / H.sum(), np.nan)
    return ax.pcolormesh(xe, ye, H.T, cmap="Greys", norm=LogNorm(vmin=1e-4, vmax=np.nanmax(H)), shading="flat", zorder=0)


def obs_points(ax, ycol, ylo=None, yhi=None, xerr=True):
    for _, o in obs.iterrows():
        y = o[ycol]
        if not (np.isfinite(o.D_Delta) and np.isfinite(y)):
            continue
        xe = [[max(o.D_Delta - o.D_lo, 0)], [max(o.D_hi - o.D_Delta, 0)]] if xerr and np.isfinite(o.D_lo) else None
        ye = ([[max(y - o[ylo], 0)], [max(o[yhi] - y, 0)]] if ylo and np.isfinite(o[ylo]) and np.isfinite(o[yhi]) else None)
        a = QA.get(o.quality, 0.5)
        ax.errorbar(o.D_Delta, y, xerr=xe, yerr=ye, fmt=MK.get(o.method, "o"), color=REG_COL.get(o.regime, "k"),
                    ms=7, mec="k", mew=0.5, alpha=a, elinewidth=0.8, capsize=1.5, zorder=3)


XLIM, YLIM = (10, 300), (-0.6, 1.5)
fig, axs = plt.subplots(2, 2, figsize=(14, 11), sharex=True, sharey="row")
for j, (name, colr) in enumerate(MODELS.items()):
    m = M[name]
    for i, (ycol, mk, ylo, yhi, ylab) in enumerate([
            ("S100", "S100", "S100_lo", "S100_hi", "S(100 nm)"),
            ("S_at_D", "S_DSB", "S_lo", "S_hi", "S(D$_{SB}$)")]):
        ax = axs[i, j]
        pc = density(ax, m["DSB"].ravel(), m[mk].ravel(), m["w"].ravel(), XLIM, YLIM)
        obs_points(ax, ycol, ylo, yhi)
        sm = site[site.model == name]
        smk = {"S100": "mod_S100", "S_DSB": "mod_S_D"}[mk]
        ax.scatter(sm.mod_D, sm[smk], s=22, facecolors="none", edgecolors=colr, lw=1.2, zorder=4)
        med = wq(m[mk].ravel(), m["w"].ravel(), (0.5,))[0], wq(m["DSB"].ravel(), m["w"].ravel(), (0.5,))[0]
        ax.plot(med[1], med[0], marker="*", ms=16, color=colr, mec="k", zorder=5)
        ax.axhline(0, color="0.6", lw=0.8); ax.axhline(1, color="0.6", lw=0.8, ls=":")
        ax.set(xscale="log", xlim=XLIM, ylim=YLIM)
        ax.xaxis.set_major_locator(FixedLocator([10, 20, 30, 50, 70, 100, 150, 200, 300]))
        ax.xaxis.set_major_formatter(ScalarFormatter()); ax.xaxis.set_minor_formatter(NullFormatter())
        ax.grid(alpha=0.25)
        if j == 0:
            ax.set_ylabel(ylab)
        if i == 1:
            ax.set_xlabel("closure diameter D$_{SB}$ = D$_\\Delta$ (nm)")
        if i == 0:
            ax.set_title(f"{name} grid cells (grey) vs observations", fontsize=11)
fig.colorbar(pc, ax=axs, shrink=0.6, pad=0.01, label="fraction of model area (log)")
h = ([Line2D([], [], ls="", marker="o", color=c, mec="k", ms=7, label=k) for k, c in REG_COL.items()]
     + [Line2D([], [], ls="", marker=mk_, color="0.75", mec="k", ms=7, label=k) for k, mk_ in MK.items()]
     + [Line2D([], [], ls="", marker="o", mfc="none", mec=c, ms=6, label=f"{k} at obs location") for k, c in MODELS.items()]
     + [Line2D([], [], ls="", marker="*", color=c, mec="k", ms=12, label=f"{k} global median") for k, c in MODELS.items()])
fig.legend(handles=h, loc="lower center", ncol=5, fontsize=8, frameon=False, bbox_to_anchor=(0.45, -0.02))
fig.savefig(FIG_DIR / "model_obs_S_vs_DSB.png", dpi=150, bbox_inches="tight")
plt.show()

## 6 · Figure: model at the observed location vs the observation

One point per source and model (blue NorESM2, orange EC-Earth3); the dashed line is 1:1. Marker = method as
above; fainter = lower observational quality.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(16, 5.2))
spec = [("obs_S100", "mod_S100", "S(100 nm)", (-0.6, 1.5), False),
        ("obs_D", "mod_D", "D$_{SB}$ (nm)", (10, 300), True),
        ("obs_S_D", "mod_S_D", "S(D$_{SB}$)", (-0.6, 1.5), False)]
for ax, (xo, ym, lab, lim, logax) in zip(axs, spec):
    for name, colr in MODELS.items():
        s = site[site.model == name]
        for _, r in s.iterrows():
            ax.scatter(r[xo], r[ym], marker=MK.get(r.method, "o"), s=40, color=colr, edgecolor="k", lw=0.4,
                       alpha=QA.get(r.quality, 0.5))
        ok = np.isfinite(s[xo]) & np.isfinite(s[ym]) & s.quality.isin(["robust", "moderate"])
        if ok.sum() >= 4:
            a, b = np.log10(s[xo][ok]) if logax else s[xo][ok], np.log10(s[ym][ok]) if logax else s[ym][ok]
            rr = np.corrcoef(a, b)[0, 1]
            bias = (10 ** np.median(b - a)) if logax else np.median(b - a)
            ax.text(0.03, 0.97 - 0.08 * list(MODELS).index(name),
                    f"{name}: r = {rr:.2f}, median {'ratio' if logax else 'diff'} {bias:.2f}" , color=colr,
                    transform=ax.transAxes, va="top", fontsize=8.5)
    ax.plot(lim, lim, "k--", lw=0.8)
    ax.set(xlim=lim, ylim=lim, xlabel=f"observed {lab}", ylabel=f"model {lab}")
    if logax:
        ax.set(xscale="log", yscale="log")
    ax.grid(alpha=0.25)
fig.suptitle("Model at the observed locations vs observations (statistics: robust and moderate sources)", fontsize=11)
fig.tight_layout()
fig.savefig(FIG_DIR / "model_vs_obs_at_sites.png", dpi=150, bbox_inches="tight")
plt.show()

## 7 · Regime summary: medians of observations and of the models at the same locations

In [ ]:
t = site[site.quality.isin(["robust", "moderate"])].copy()
summary = (t.groupby(["regime", "model"])
             .agg(sources=("label", "nunique"), obs_S100=("obs_S100", "median"), mod_S100=("mod_S100", "median"),
                  obs_D=("obs_D", "median"), mod_D=("mod_D", "median"),
                  obs_S_D=("obs_S_D", "median"), mod_S_D=("mod_S_D", "median"))
             .round(2))
summary.to_csv(FIG_DIR / "model_obs_regime_summary.csv")
summary